In [0]:
-----------------------------------------QUERY PARA OBTENER CADA PRODUCTO CON SU INFORMACION DEL CATALOGO-----------------------------------------
WITH cleaned_prices AS (
    SELECT
      b.product_id,
      b.retailer,
      b.scraped_at,
      get_json_object(b.raw_data, '$.sku') as sku,
      get_json_object(b.raw_data, '$.name') as name,
      get_json_object(b.raw_data, '$.list_price') as list_price,
      get_json_object(b.raw_data, '$.cash_price') as cash_price,
      get_json_object(b.raw_data, '$.rating') as rating,
      get_json_object(b.raw_data, '$.installments') as installments,
      get_json_object(b.raw_data, '$.specification') as specifications,
      get_json_object(b.raw_data, '$.description') as description,
      get_json_object(b.raw_data, '$.stock') AS is_in_stock,
      c.brand AS catalog_brand,
      TRY_CAST(get_json_object(b.raw_data, '$.list_price') AS DOUBLE) AS clean_list,
      TRY_CAST(get_json_object(b.raw_data, '$.cash_price') AS DOUBLE) AS clean_cash
    FROM workspace.products.bronze_scraped_products b
    INNER JOIN workspace.products.catalog c 
      ON b.product_id = c.product_id AND b.retailer = c.retailer
  )
  -- Aquí procesamos los datos de Bronze antes de insertarlos
  SELECT 
    product_id,
    retailer,
    catalog_brand AS brand, -- Usamos la marca del catálogo que es más confiable
    sku,
    name,
    clean_list AS list_price,
    clean_cash AS cash_price,
    COALESCE(clean_cash, clean_list) AS effective_price,
    scraped_at,
    CAST(scraped_at AS DATE) AS scraped_date,
    ROUND((1 - (clean_cash / clean_list)) * 100, 2) AS discount_pct,
    installments AS installments_json,
    specifications AS specification_json,
    description,
    CASE 
      WHEN rating IS NULL OR rating = '{}' OR rating = '' THEN 0 
      ELSE TRY_CAST(get_json_object(rating, '$.ratingValue') AS DOUBLE)
    END AS rating_value,
    is_in_stock AS stock,
    -- Columnas computadas (de celdas 8-14)
    (clean_list IS NOT NULL OR clean_cash IS NOT NULL) AS is_available,
    (name IS NOT NULL AND name != '') AS is_valid_name,
    CASE 
      WHEN clean_list IS NULL AND clean_cash IS NULL THEN FALSE
      -- Umbral actualizado a 20 Millones para productos de alta gama
      WHEN clean_list > 20000000 OR clean_list < 0 THEN FALSE
      WHEN clean_cash > 20000000 OR clean_cash < 0 THEN FALSE
      -- Validación de gap (mantiene el límite de 500% para detectar anomalías)
      WHEN ((ABS(clean_list - clean_cash) / NULLIF(LEAST(clean_list, clean_cash), 0)) * 100) > 500 THEN FALSE
      ELSE TRUE 
    END AS is_valid_price,
    CASE 
      WHEN TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) = 0 THEN '0'
      ELSE CONCAT(ROUND((1 - (TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2), '%')
    END AS discount_applied_str,
    CASE 
      WHEN TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) = 0 THEN 0.0
      ELSE ROUND((1 - (TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2)
    END AS discount_applied_int,
    CASE 
      WHEN installments IS NULL OR installments = '{}' OR installments = '' THEN FALSE 
      ELSE TRUE 
    END AS has_installments,
    CASE 
      WHEN description IS NULL OR description = '' THEN FALSE 
      ELSE TRUE 
    END AS has_description
  FROM cleaned_prices;



SELECT COUNT(*) FROM products.silver_products;
SELECT * FROM products.silver_products ORDER BY scraped_at DESC;

DESCRIBE TABLE products.silver_products;

-----------------------------------------QUERY PARA OBTENER CADA PRODUCTO CON SU MAXIMA ACTUALIZACION DE PRECIOS Y STOCK-----------------------------------------
WITH max_scraped_at AS (
  SELECT
    catal.product_id,
    catal.brand,
    get_json_object(scrap.raw_data, '$.brand') AS scraped_brand,
    get_json_object(scrap.raw_data, '$.sku') AS scraped_sku,
    get_json_object(scrap.raw_data, '$.name') AS scraped_name,
    catal.retailer,
    catal.main_category,
    catal.sub_category,
    get_json_object(scrap.raw_data, '$.main_category') AS scraped_main_category,
    get_json_object(scrap.raw_data, '$.sub_category') AS scraped_sub_category,
    TRY_CAST(REGEXP_REPLACE(get_json_object(scrap.raw_data, '$.list_price'), '[\\$.]', '') AS DOUBLE) AS scraped_list_price,
    TRY_CAST(REGEXP_REPLACE(get_json_object(scrap.raw_data, '$.cash_price'), '[\\$.]', '') AS DOUBLE) AS scraped_cash_price,
    CASE 
      WHEN get_json_object(scrap.raw_data, '$.sku') IS NULL OR get_json_object(scrap.raw_data, '$.sku') = '' 
           OR (
                (get_json_object(scrap.raw_data, '$.list_price') IS NULL OR get_json_object(scrap.raw_data, '$.list_price') = '') 
                AND (get_json_object(scrap.raw_data, '$.cash_price') IS NULL OR get_json_object(scrap.raw_data, '$.cash_price') = '')
              )
      THEN "False"
      ELSE "True"
    END AS is_available,
    scrap.scraped_at AS updated_at,
    catal.link,
    ROW_NUMBER() OVER (PARTITION BY scrap.product_id, scrap.retailer ORDER BY scrap.scraped_at DESC) AS rn
  FROM workspace.products.bronze_scraped_products scrap
  JOIN workspace.products.catalog catal
    ON scrap.product_id = catal.product_id 
   AND scrap.retailer = catal.retailer
)
SELECT * FROM max_scraped_at WHERE rn = 1 ORDER BY updated_at DESC;

SELECT 
  b.product_id,
  b.retailer,
  b.scraped_at,
  COUNT(*) as cnt
FROM workspace.products.bronze_scraped_products b
INNER JOIN workspace.products.catalog c 
  ON b.product_id = c.product_id 
 AND b.retailer = c.retailer
GROUP BY 1,2,3
HAVING COUNT(*) > 1;

SELECT 
  b.product_id,
  b.retailer,
  b.scraped_at,
  COUNT(*) as cnt
FROM workspace.products.bronze_scraped_products b
INNER JOIN workspace.products.catalog c 
  ON b.product_id = c.product_id 
 AND b.retailer = c.retailer
GROUP BY 1,2,3
HAVING COUNT(*) > 1;